本 notebook 是 [M05.6 序列模型](../notes/06-sequence-models.qmd) 的实例讲解。

1. 1. 手写 RNN 递推与框架对齐
2. 2. LSTM 与 GRU 门控的精确约定
3. 3. 长链与截断 BPTT
4. 4. 变长序列与 padding
5. 5. 合成 AR 预测与强简单基线

在 VS Code 中选择仓库 `.venv` 内核，逐单元运行；运行结果随文件保存。示例使用固定随机种子和合成数据。

运行前安装已声明依赖：`uv sync --locked --extra perf --extra data --extra dl`。本例使用 CPU，不需外部数据或预训练权重。

In [1]:
import numpy as np
import torch
from matplotlib_inline.backend_inline import set_matplotlib_formats
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_sequence
from torch.utils.data import DataLoader, TensorDataset

from semiai.plotting import use_style

use_style()
set_matplotlib_formats("svg")
rng = np.random.default_rng(42)

## 1. 手写 RNN 递推与框架对齐

显式按时间循环，使用框架的权重和两个偏置，核验所有时刻输出与最终隐藏状态。

In [2]:
torch.set_num_threads(1)
torch.manual_seed(42)
x = torch.randn(2, 5, 3, dtype=torch.float64)
model = nn.RNN(3, 4, batch_first=True).double()
h = torch.zeros(2, 4, dtype=torch.float64)
history = []
for t in range(x.shape[1]):
    h = torch.tanh(
        x[:, t] @ model.weight_ih_l0.T
        + model.bias_ih_l0
        + h @ model.weight_hh_l0.T
        + model.bias_hh_l0
    )
    history.append(h)
manual = torch.stack(history, dim=1)
output, last = model(x)
print(
    "全序列最大差:",
    (manual - output).abs().max().item(),
    "最终状态差:",
    (h - last[0]).abs().max().item(),
)
assert torch.allclose(manual, output, atol=1e-12)
assert torch.allclose(h, last[0], atol=1e-12)

全序列最大差: 1.1102230246251565e-16 最终状态差: 1.1102230246251565e-16


读法：batch_first 只改变输入输出的 batch/时间布局，隐藏状态仍按层×方向、batch、hidden 排列；共享同一组参数是 RNN 的核心。

## 2. LSTM 与 GRU 门控的精确约定

手工拆出框架的门顺序，核验单步状态；特别检查 PyTorch GRU 的 reset 在隐状态线性变换之后作用。

In [3]:
x = torch.randn(2, 3, dtype=torch.float64)
h = torch.randn(2, 4, dtype=torch.float64)
c = torch.randn(2, 4, dtype=torch.float64)
lstm = nn.LSTMCell(3, 4).double()
gates = nn.functional.linear(x, lstm.weight_ih, lstm.bias_ih) + nn.functional.linear(
    h, lstm.weight_hh, lstm.bias_hh
)
i, f, g, o = gates.chunk(4, dim=1)
i = i.sigmoid()
f = f.sigmoid()
g = g.tanh()
o = o.sigmoid()
c_manual = f * c + i * g
h_manual = o * c_manual.tanh()
h_ref, c_ref = lstm(x, (h, c))
print(
    "LSTM h/c 最大差:", (h_manual - h_ref).abs().max().item(), (c_manual - c_ref).abs().max().item()
)
assert torch.allclose(h_manual, h_ref, atol=1e-12) and torch.allclose(c_manual, c_ref, atol=1e-12)
gru = nn.GRUCell(3, 4).double()
ir, iz, inew = nn.functional.linear(x, gru.weight_ih, gru.bias_ih).chunk(3, 1)
hr, hz, hnew = nn.functional.linear(h, gru.weight_hh, gru.bias_hh).chunk(3, 1)
r = (ir + hr).sigmoid()
z = (iz + hz).sigmoid()
new = (inew + r * hnew).tanh()
h_manual = (1 - z) * new + z * h
h_ref = gru(x, h)
print("GRU 最大差:", (h_manual - h_ref).abs().max().item())
assert torch.allclose(h_manual, h_ref, atol=1e-12)

LSTM h/c 最大差: 0.0 0.0
GRU 最大差: 2.220446049250313e-16


读法：LSTM 提供加性 cell 路径，GRU 将保留与候选状态混合；不同文献与库的 reset 位置、门顺序和更新门含义可能不同，不能只凭名字复制公式。

## 3. 长链与截断 BPTT

在可解析线性递推上核验梯度幂次，再在中间 detach 状态，检查早期初值不再收到梯度。

In [4]:
for a in [0.8, 0.98, 1.05]:
    h0 = torch.tensor(1.0, dtype=torch.float64, requires_grad=True)
    h = h0
    for _ in range(100):
        h = a * h
    gradient = torch.autograd.grad(h, h0)[0]
    print("a", a, "AD 梯度", gradient.item(), "解析", a**100)
    assert np.isclose(gradient.item(), a**100, rtol=1e-12)
h0 = torch.tensor(1.0, requires_grad=True)
a = torch.tensor(0.98, requires_grad=True)
h = h0
for t in range(100):
    if t == 50:
        h = h.detach()
    h = a * h
print("截断后对初值的梯度:", torch.autograd.grad(h, h0, allow_unused=True)[0])

a 0.8 AD 梯度 2.0370359763344983e-10 解析 2.0370359763344975e-10
a 0.98 AD 梯度 0.132619555894753 解析 0.13261955589475294
a 1.05 AD 梯度 131.50125784630401 解析 131.50125784630401
截断后对初值的梯度: None


读法：detach 保留状态数值但截断过去的导数路径，因此 truncated BPTT 改变可学习的信用分配跨度，不只是减少内存。门控路径仍可能长期遗忘。

## 4. 变长序列与 padding

同一 GRU 分别处理真实长度序列与打包 batch，核验最终状态一致，避免直接取补零后的最后位置。

In [5]:
sequences = [torch.randn(length, 2) for length in [5, 2, 4]]
lengths = torch.tensor([len(s) for s in sequences])
padded = pad_sequence(sequences, batch_first=True)
model = nn.GRU(2, 6, batch_first=True).eval()
packed = pack_padded_sequence(padded, lengths, batch_first=True, enforce_sorted=False)
with torch.no_grad():
    _, packed_h = model(packed)
    individual = torch.cat([model(s[None])[1] for s in sequences], dim=1)
    padded_output, padded_h = model(padded)
print("packed 与逐条最终状态差:", (packed_h - individual).abs().max().item())
print("直接补零后的最终状态差:", (padded_h - individual).abs().max().item())
assert torch.allclose(packed_h, individual, atol=1e-6)
print(
    "长度:",
    lengths.tolist(),
    "padded shape:",
    tuple(padded.shape),
    "output shape:",
    tuple(padded_output.shape),
)

packed 与逐条最终状态差: 7.450580596923828e-09
直接补零后的最终状态差: 0.3804081678390503
长度: [5, 2, 4] padded shape: (3, 5, 2) output shape: (3, 5, 6)


读法：padding 不是无操作，递推会继续改变状态；packing 使用真实长度排除无效步。序列到序列损失也必须用有效位置 mask，避免短序列被大量补零主导。

## 5. 合成 AR 预测与强简单基线

独立生成训练和评价序列，比较 persistence、线性一步预测与小 GRU；该生成器本来是一阶 Markov，因此长记忆模型没有必然优势。

In [6]:
def make_series(count, seed):
    local = np.random.default_rng(seed)
    series = np.empty((count, 14))
    series[:, 0] = local.normal(0, 0.2 / np.sqrt(1 - 0.8**2), count)
    for t in range(1, 14):
        series[:, t] = 0.8 * series[:, t - 1] + local.normal(0, 0.2, count)
    return torch.tensor(series[:, :-1, None], dtype=torch.float32), torch.tensor(
        series[:, -1, None], dtype=torch.float32
    )


train_x, train_y = make_series(300, 1)
test_x, test_y = make_series(180, 2026)
coef = np.linalg.lstsq(train_x[:, -1].numpy(), train_y.numpy(), rcond=None)[0]
linear = torch.from_numpy(test_x[:, -1].numpy() @ coef)
torch.manual_seed(42)
recurrent = nn.GRU(1, 10, batch_first=True)
head = nn.Linear(10, 1)
optimizer = torch.optim.Adam([*recurrent.parameters(), *head.parameters()], lr=0.01)
loader = DataLoader(
    TensorDataset(train_x, train_y),
    batch_size=50,
    shuffle=True,
    generator=torch.Generator().manual_seed(42),
)
for _ in range(20):
    for x, y in loader:
        optimizer.zero_grad(set_to_none=True)
        output, _ = recurrent(x)
        prediction = head(output[:, -1])
        loss = nn.functional.mse_loss(prediction, y)
        loss.backward()
        optimizer.step()
recurrent.eval()
head.eval()
with torch.no_grad():
    output, _ = recurrent(test_x)
    prediction = head(output[:, -1])
for name, pred in [
    ("persistence", test_x[:, -1]),
    ("linear", linear),
    ("GRU", prediction),
    ("known oracle", 0.8 * test_x[:, -1]),
]:
    print(name, "独立序列 MSE:", nn.functional.mse_loss(pred, test_y).item())
print("已知不可约噪声方差:", 0.2**2, "拟合一步系数:", coef.ravel())

persistence 独立序列 MSE: 0.042440857738256454
linear 独立序列 MSE: 0.03810972347855568
GRU 独立序列 MSE: 0.03873888775706291
known oracle 独立序列 MSE: 0.03812336549162865
已知不可约噪声方差: 0.04000000000000001 拟合一步系数: [0.82473946]


读法：有限测试 MSE 会围绕噪声方差波动；已知线性条件均值是强基线，小 GRU 的训练只展示接口与比较过程，不预设更复杂模型胜出。

## 小结

RNN 通过时间共享参数建立递推，LSTM 与 GRU 用门控改善信息保留，BPTT 决定信用如何沿历史传播。形状、真实长度与预测方向是实现契约；简单基线、独立序列和滚动评价决定模型是否提供了任务所需的额外能力。